### **DimUser**

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_pth = (os.path.join(os.getcwd(),'..','..'))
sys.path.append(project_pth)
import utils.transformations as reusable


### *AUTOLOADER*

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimUser/checkpoint/")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimUser")

In [0]:
display(df_user)

In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
from utils.transformations import reusable

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
# Read as batch to preview in table format
df_user_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimUser")
display(df_user_preview)

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimUser/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectt.dfs.core.windows.net/DimUser/data")\
.toTable("spotify_cata.silver.DimUser")
   

In [0]:
display(spark.read.format("delta").load("abfss://silver@storageazureprojectt.dfs.core.windows.net/DimUser/data"))

### **DimArtist**

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimArtist/checkpoint/")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimArtist")

In [0]:
display(df_artist)

In [0]:
# Read as batch to preview in table format
df_artist_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimArtist")
display(df_artist_preview)

In [0]:
df_artist_obj = reusable()

df_artist = df_artist_obj.dropColumns(df_artist,['_rescued_data'])
df_artist = df_artist.dropDuplicates(['artist_id'])
display(df_artist)

In [0]:
# Read as batch to preview in table format
df_artist_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimArtist")
display(df_artist_preview)

In [0]:
df_artist.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimArtist/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectt.dfs.core.windows.net/DimArtist/data")\
.toTable("spotify_cata.silver.DimArtist")

In [0]:
display(spark.read.format("delta").load("abfss://silver@storageazureprojectt.dfs.core.windows.net/DimArtist/data"))

### **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimTrack/checkpoint/")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track)

In [0]:
df_track_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimTrack")
display(df_track_preview)

In [0]:
df_track = df_track.withColumn("durationFlag", when(col('duration_sec')<150,"low")\
                                              .when(col('duration_sec')<300,"medium")\
                                                .otherwise("high"))

df_track = df_track.withColumn("track_name", regexp_replace(col('track_name'), '-', ' '))
display(df_track)

In [0]:

df_track_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimTrack")
display(df_track_preview)



In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimTrack/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectt.dfs.core.windows.net/DimTrack/data")\
.toTable("spotify_cata.silver.DimTrack")

In [0]:

# checking the preview 
df_track_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimTrack")
df_track_preview = df_track_preview.withColumn("durationFlag", when(col('duration_sec')<150,"low")\
                                              .when(col('duration_sec')<300,"medium")\
                                                .otherwise("high"))
display(df_track_preview)


df_track_preview = spark.read.parquet("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimTrack")
df_track_preview = df_track_preview.withColumn("track_name", regexp_replace(col('track_name'), '-', ' '))
display(df_track_preview)

### **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimDate/checkpoint/")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageazureprojectt.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])

df_date.writeStream.format("delta")

df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimDate/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectt.dfs.core.windows.net/DimDate/data")\
.toTable("spotify_cata.silver.DimDate")

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/DimDate/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectt.dfs.core.windows.net/DimDate/data")\
.toTable("spotify_cata.silver.DimDate")

### **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/FactStream/checkpoint/")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@storageazureprojectt.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact)

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream.format("delta")

df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectt.dfs.core.windows.net/FactStream/checkpoint/")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectt.dfs.core.windows.net/FactStream/data")\
.toTable("spotify_cata.silver.FactStream")